# L24 Representation Drift versus Depth Degradation

This notebook tests whether adjacent-frame drift in the full L24 representation identifies frames on which forced shallow inference causes greater task-level degradation. The primary endpoint is a configurable per-frame matched-prediction surrogate; it is **not** a decomposition of nuScenes mAP or NDS.

Production inputs must contain counterfactual L8/L12/L16/L24 predictions evaluated from the same pre-frame temporal-memory state. Set `DRIFT_DEPTH_SYNTHETIC=1` only for the lightweight execution check.

In [8]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr
from sklearn.isotonic import IsotonicRegression

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd()
SYNTHETIC = os.getenv('DRIFT_DEPTH_SYNTHETIC', '0') == '1'
OUTPUT_DIR = Path(os.getenv('DRIFT_DEPTH_OUTPUT_DIR', ROOT / 'outputs/drift_depth_analysis'))
PAPER_DIR = Path(os.getenv('DRIFT_DEPTH_PAPER_DIR', ROOT / 'paper/figures'))

CFG = {
    'manifest': ROOT / 'outputs/drift_depth_analysis/counterfactual_manifest.json',
    'drift_csv': ROOT / 'outputs/policy_signal_stats/val/policy_feature_similarity.csv',
    'data_root': ROOT / 'data/nuscenes',
    'version': 'v1.0-trainval',
    'loss_csv': ROOT / 'outputs/drift_depth_analysis/per_frame_detection_loss.csv',
    'depths': ('L8', 'L12', 'L16', 'L24'),
    'score_floor': 0.10,
    'center_gate_m': 4.0,
    'velocity_scale_mps': 10.0,
    'weights': {'confidence': 0.5, 'center': 0.2, 'size': 0.1, 'yaw': 0.1, 'velocity': 0.1},
    'false_positive_penalty': 1.0,
    'false_negative_penalty': 1.0,
    'quantiles': 5,
    'bootstrap_repetitions': 2000,
    'seed': 20260804,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not SYNTHETIC:
    PAPER_DIR.mkdir(parents=True, exist_ok=True)
CFG

{'manifest': PosixPath('/home/moshekimhi_google_com/StreamPETR/outputs/drift_depth_analysis/counterfactual_manifest.json'),
 'drift_csv': PosixPath('/home/moshekimhi_google_com/StreamPETR/outputs/policy_signal_stats/val/policy_feature_similarity.csv'),
 'data_root': PosixPath('/home/moshekimhi_google_com/StreamPETR/data/nuscenes'),
 'version': 'v1.0-trainval',
 'loss_csv': PosixPath('/home/moshekimhi_google_com/StreamPETR/outputs/drift_depth_analysis/per_frame_detection_loss.csv'),
 'depths': ('L8', 'L12', 'L16', 'L24'),
 'score_floor': 0.1,
 'center_gate_m': 4.0,
 'velocity_scale_mps': 10.0,
 'weights': {'confidence': 0.5,
  'center': 0.2,
  'size': 0.1,
  'yaw': 0.1,
  'velocity': 0.1},
 'false_positive_penalty': 1.0,
 'false_negative_penalty': 1.0,
 'quantiles': 5,
 'bootstrap_repetitions': 2000,
 'seed': 20260804}

## Input contract

The manifest must declare `shared_pre_frame_state: true`, a run ID, the canonical state-advance action, and one official nuScenes `results_nusc.json` path per depth. All four files must cover exactly the same sample tokens.

In [9]:
def load_json(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    return json.loads(path.read_text())


def validate_manifest(manifest):
    required = {'shared_pre_frame_state', 'run_id', 'canonical_state_advance_action', 'prediction_paths'}
    missing = required - manifest.keys()
    if missing:
        raise ValueError(f'manifest missing keys: {sorted(missing)}')
    if manifest['shared_pre_frame_state'] is not True:
        raise ValueError('counterfactual predictions must share pre-frame state')
    paths = manifest['prediction_paths']
    if set(paths) != set(CFG['depths']):
        raise ValueError(f"prediction_paths must contain {CFG['depths']}")
    return {
        depth: Path(path) if Path(path).is_absolute() else ROOT / path
        for depth, path in paths.items()
    }


def standardize_prediction(record):
    return {
        'name': record['detection_name'],
        'translation': np.asarray(record['translation'], dtype=float),
        'size': np.asarray(record['size'], dtype=float),
        'rotation': np.asarray(record['rotation'], dtype=float),
        'velocity': np.asarray(record.get('velocity', [0.0, 0.0])[:2], dtype=float),
        'score': float(record['detection_score']),
    }


def load_predictions(path):
    payload = load_json(path)
    if not isinstance(payload.get('results'), dict):
        raise ValueError(f'{path} is not a nuScenes results JSON')
    return {
        token: [standardize_prediction(record) for record in records
                if float(record['detection_score']) >= CFG['score_floor']]
        for token, records in payload['results'].items()
    }


def validate_token_coverage(predictions):
    coverage = {depth: set(records) for depth, records in predictions.items()}
    reference = coverage['L24']
    mismatched = {depth: len(tokens ^ reference)
                  for depth, tokens in coverage.items() if tokens != reference}
    if mismatched:
        raise ValueError(f'depth token coverage differs from L24: {mismatched}')
    if not reference:
        raise ValueError('prediction artifacts contain no sample tokens')
    return reference


def load_drift(path, tokens):
    frame = pd.read_csv(path)
    required = {'token', 'scene_token', 'has_previous_frame', 'similarity_L24_mean'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'drift table missing columns: {sorted(missing)}')
    frame = frame[frame['token'].isin(tokens)].copy()
    if frame['token'].duplicated().any():
        raise ValueError('drift table contains duplicate sample tokens')
    frame = frame[(frame['has_previous_frame'] == 1) & frame['similarity_L24_mean'].notna()].copy()
    frame['l24_drift'] = 1.0 - frame['similarity_L24_mean'].astype(float)
    if not np.isfinite(frame['l24_drift']).all():
        raise ValueError('L24 drift contains non-finite values')
    if frame.empty:
        raise ValueError('no prediction tokens have valid adjacent-frame L24 drift')
    return frame[['token', 'scene_token', 'l24_drift']]


def load_ground_truth(data_root, version, tokens):
    from nuscenes import NuScenes
    from nuscenes.eval.detection.utils import category_to_detection_name

    nusc = NuScenes(version=version, dataroot=str(data_root), verbose=False)
    ground_truth = {}
    for token in tokens:
        boxes = []
        for annotation_token in nusc.get('sample', token)['anns']:
            annotation = nusc.get('sample_annotation', annotation_token)
            name = category_to_detection_name(annotation['category_name'])
            if name is None:
                continue
            boxes.append({
                'name': name,
                'translation': np.asarray(annotation['translation'], dtype=float),
                'size': np.asarray(annotation['size'], dtype=float),
                'rotation': np.asarray(annotation['rotation'], dtype=float),
                'velocity': np.nan_to_num(nusc.box_velocity(annotation_token)[:2]),
                'score': 1.0,
            })
        ground_truth[token] = boxes
    return ground_truth

## Per-frame matched-prediction error

Matching is class-aware and uses a configurable center-distance gate. Matched costs combine confidence, center, size, yaw, and velocity errors; unmatched predictions and ground-truth boxes receive explicit penalties. The box-only sensitivity endpoint removes confidence but retains geometry and unmatched-box penalties.

In [10]:
def quaternion_yaw(rotation):
    w, x, y, z = rotation
    return np.arctan2(2.0 * (w * z + x * y), 1.0 - 2.0 * (y * y + z * z))


def wrapped_angle_error(a, b):
    return abs(np.arctan2(np.sin(a - b), np.cos(a - b))) / np.pi


def pair_components(prediction, target):
    center_m = np.linalg.norm(prediction['translation'][:2] - target['translation'][:2])
    return {
        'confidence': 1.0 - prediction['score'],
        'center': center_m / CFG['center_gate_m'],
        'size': np.mean(np.abs(np.log(
            np.clip(prediction['size'], 1e-6, None) / np.clip(target['size'], 1e-6, None)))),
        'yaw': wrapped_angle_error(quaternion_yaw(prediction['rotation']), quaternion_yaw(target['rotation'])),
        'velocity': np.linalg.norm(prediction['velocity'] - target['velocity']) / CFG['velocity_scale_mps'],
    }


def frame_error(predictions, targets, include_confidence=True):
    denominator = max(len(predictions), len(targets), 1)
    total = 0.0
    matched_predictions, matched_targets = set(), set()
    for name in sorted({box['name'] for box in predictions + targets}):
        pred_indices = [i for i, box in enumerate(predictions) if box['name'] == name]
        target_indices = [i for i, box in enumerate(targets) if box['name'] == name]
        if not pred_indices or not target_indices:
            continue
        matrix = np.full((len(pred_indices), len(target_indices)), 1e6)
        for row, pred_index in enumerate(pred_indices):
            for column, target_index in enumerate(target_indices):
                values = pair_components(predictions[pred_index], targets[target_index])
                if values['center'] <= 1.0:
                    keys = CFG['weights'] if include_confidence else [
                        key for key in CFG['weights'] if key != 'confidence']
                    matrix[row, column] = sum(CFG['weights'][key] * values[key] for key in keys)
        rows, columns = linear_sum_assignment(matrix)
        for row, column in zip(rows, columns):
            if matrix[row, column] >= 1e6:
                continue
            total += matrix[row, column]
            matched_predictions.add(pred_indices[row])
            matched_targets.add(target_indices[column])
    total += CFG['false_positive_penalty'] * (len(predictions) - len(matched_predictions))
    total += CFG['false_negative_penalty'] * (len(targets) - len(matched_targets))
    return total / denominator


def per_frame_errors(drift_frame, ground_truth, predictions):
    rows = []
    for record in drift_frame.itertuples(index=False):
        row = {'token': record.token, 'scene_token': record.scene_token, 'l24_drift': record.l24_drift}
        for depth in CFG['depths']:
            boxes, targets = predictions[depth][record.token], ground_truth[record.token]
            row[f'error_{depth}'] = frame_error(boxes, targets, True)
            row[f'box_error_{depth}'] = frame_error(boxes, targets, False)
        for depth in CFG['depths'][:-1]:
            row[f'degradation_{depth}'] = row[f'error_{depth}'] - row['error_L24']
            row[f'box_degradation_{depth}'] = row[f'box_error_{depth}'] - row['box_error_L24']
        rows.append(row)
    return pd.DataFrame(rows)

In [11]:
def synthetic_inputs():
    rng = np.random.default_rng(CFG['seed'])
    tokens = [f'scene{s}_frame{f}' for s in range(3) for f in range(1, 11)]
    scenes = [f'scene{s}' for s in range(3) for _ in range(10)]
    drift = np.linspace(0.01, 0.30, len(tokens))
    drift_frame = pd.DataFrame({'token': tokens, 'scene_token': scenes, 'l24_drift': drift})
    ground_truth, predictions = {}, {depth: {} for depth in CFG['depths']}
    scales = {'L8': 3.0, 'L12': 2.0, 'L16': 1.0, 'L24': 0.15}
    for token, value in zip(tokens, drift):
        target = {
            'name': 'car', 'translation': np.array([0.0, 0.0, 0.0]),
            'size': np.array([2.0, 4.0, 1.5]), 'rotation': np.array([1.0, 0.0, 0.0, 0.0]),
            'velocity': np.array([0.0, 0.0]), 'score': 1.0,
        }
        ground_truth[token] = [target]
        for depth, scale in scales.items():
            offset = scale * value + rng.normal(0.0, 0.005)
            predictions[depth][token] = [{
                **target, 'translation': np.array([offset, 0.0, 0.0]),
                'score': float(np.clip(0.98 - 0.2 * scale * value, 0.1, 1.0)),
            }]
    return drift_frame, ground_truth, predictions


if SYNTHETIC:
    _drift, _gt, _predictions = synthetic_inputs()
    _target, _prediction = _gt[_drift.iloc[0]['token']], _predictions['L8'][_drift.iloc[0]['token']]
    assert frame_error([], [], True) == 0.0
    assert frame_error([], _target, True) == 1.0
    assert frame_error(_prediction, [], True) == 1.0
    _errors = per_frame_errors(_drift, _gt, _predictions)
    assert (_errors['degradation_L8'].mean() > _errors['degradation_L12'].mean() >
            _errors['degradation_L16'].mean())

## Drift quantiles and scene-grouped uncertainty

Frames are assigned to five equal-frequency L24-drift bins. Confidence intervals resample complete scenes, preserving within-scene dependence.

In [12]:
def assign_quantiles(frame):
    frame = frame.copy()
    if frame['scene_token'].nunique() < 2:
        raise ValueError('analysis requires at least two scenes')
    if frame['l24_drift'].nunique() < CFG['quantiles']:
        raise ValueError('too few distinct drift values for requested quantiles')
    frame['drift_quantile'] = pd.qcut(
        frame['l24_drift'], CFG['quantiles'], labels=False, duplicates='raise') + 1
    return frame


def scene_bootstrap(frame, value_columns):
    rng = np.random.default_rng(CFG['seed'])
    scenes = frame['scene_token'].unique()
    quantiles = sorted(frame['drift_quantile'].unique())
    samples = {(column, quantile): [] for column in value_columns for quantile in quantiles}
    correlations = {column: [] for column in value_columns}
    for _ in range(CFG['bootstrap_repetitions']):
        selected = rng.choice(scenes, size=len(scenes), replace=True)
        resampled = pd.concat(
            [frame[frame['scene_token'] == scene] for scene in selected], ignore_index=True)
        for column in value_columns:
            for quantile in quantiles:
                values = resampled.loc[resampled['drift_quantile'] == quantile, column]
                samples[(column, quantile)].append(values.mean())
            correlations[column].append(
                spearmanr(resampled['l24_drift'], resampled[column]).statistic)
    return samples, correlations


def summarize_endpoint(frame, prefix='degradation'):
    columns = [f'{prefix}_{depth}' for depth in CFG['depths'][:-1]]
    samples, correlations = scene_bootstrap(frame, columns)
    quantile_rows, correlation_rows = [], []
    for column in columns:
        depth = column.rsplit('_', 1)[-1]
        for quantile, group in frame.groupby('drift_quantile'):
            bootstrap = np.asarray(samples[(column, quantile)])
            quantile_rows.append({
                'endpoint': prefix, 'depth': depth, 'drift_quantile': int(quantile),
                'drift_mean': group['l24_drift'].mean(), 'degradation_mean': group[column].mean(),
                'ci_low': np.nanpercentile(bootstrap, 2.5),
                'ci_high': np.nanpercentile(bootstrap, 97.5), 'frames': len(group),
            })
        observed = spearmanr(frame['l24_drift'], frame[column]).statistic
        bootstrap = np.asarray(correlations[column])
        correlation_rows.append({
            'endpoint': prefix, 'depth': depth, 'spearman': observed,
            'ci_low': np.nanpercentile(bootstrap, 2.5),
            'ci_high': np.nanpercentile(bootstrap, 97.5),
        })
    return pd.DataFrame(quantile_rows), pd.DataFrame(correlation_rows)

In [13]:
DEPTH_COLORS = {'L8': '#D55E00', 'L12': '#0072B2', 'L16': '#009E73'}


def save_figure(fig, output_stem):
    output_stem = Path(output_stem)
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_stem.with_suffix('.pdf'), bbox_inches='tight')
    fig.savefig(output_stem.with_suffix('.png'), dpi=220, bbox_inches='tight')
    plt.close(fig)


def plot_quantiles(summary, correlations, output_stem, ylabel):
    fig, ax = plt.subplots(figsize=(6.8, 4.3))
    for depth in CFG['depths'][:-1]:
        rows = summary[summary['depth'] == depth].sort_values('drift_quantile')
        rho = correlations.loc[correlations['depth'] == depth, 'spearman'].iloc[0]
        yerr = np.vstack([rows['degradation_mean'] - rows['ci_low'],
                          rows['ci_high'] - rows['degradation_mean']])
        ax.errorbar(rows['drift_quantile'], rows['degradation_mean'], yerr=yerr,
                    marker='o', capsize=3, color=DEPTH_COLORS[depth],
                    label=f'{depth} (rho={rho:.2f})')
    ax.axhline(0.0, color='0.35', linewidth=1, linestyle='--')
    ax.set(xlabel='L24 representation-drift quantile (low to high)', ylabel=ylabel,
           xticks=range(1, CFG['quantiles'] + 1))
    ax.grid(alpha=0.2)
    ax.legend(frameon=False)
    fig.tight_layout()
    save_figure(fig, output_stem)


def plot_scatter(frame, summary, output_stem):
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8), sharex=True, sharey=True)
    for ax, depth in zip(axes, CFG['depths'][:-1]):
        column = f'degradation_{depth}'
        ordered = frame.sort_values('l24_drift')
        ax.scatter(ordered['l24_drift'], ordered[column], s=7, alpha=0.16,
                   color=DEPTH_COLORS[depth], rasterized=True)
        model = IsotonicRegression(out_of_bounds='clip').fit(ordered['l24_drift'], ordered[column])
        ax.plot(ordered['l24_drift'], model.predict(ordered['l24_drift']),
                color='black', linewidth=1.5, label='isotonic trend')
        points = summary[summary['depth'] == depth].sort_values('drift_quantile')
        ax.plot(points['drift_mean'], points['degradation_mean'], 'o-',
                color=DEPTH_COLORS[depth], linewidth=1.5, label='quantile mean')
        ax.axhline(0.0, color='0.35', linewidth=1, linestyle='--')
        ax.set_title(depth)
        ax.grid(alpha=0.2)
    axes[0].set_ylabel('Matched-prediction degradation versus L24')
    axes[1].set_xlabel('Continuous L24 representation drift')
    axes[-1].legend(frameon=False)
    fig.tight_layout()
    save_figure(fig, output_stem)

## Run paired analysis

Production mode intentionally fails if the manifest or predictions are absent. It never substitutes synthetic evidence for paper results.

In [15]:
CFG

{'manifest': PosixPath('/home/moshekimhi_google_com/StreamPETR/outputs/drift_depth_analysis/counterfactual_manifest.json'),
 'drift_csv': PosixPath('/home/moshekimhi_google_com/StreamPETR/outputs/policy_signal_stats/val/policy_feature_similarity.csv'),
 'data_root': PosixPath('/home/moshekimhi_google_com/StreamPETR/data/nuscenes'),
 'version': 'v1.0-trainval',
 'loss_csv': PosixPath('/home/moshekimhi_google_com/StreamPETR/outputs/drift_depth_analysis/per_frame_detection_loss.csv'),
 'depths': ('L8', 'L12', 'L16', 'L24'),
 'score_floor': 0.1,
 'center_gate_m': 4.0,
 'velocity_scale_mps': 10.0,
 'weights': {'confidence': 0.5,
  'center': 0.2,
  'size': 0.1,
  'yaw': 0.1,
  'velocity': 0.1},
 'false_positive_penalty': 1.0,
 'false_negative_penalty': 1.0,
 'quantiles': 5,
 'bootstrap_repetitions': 2000,
 'seed': 20260804}

In [14]:
if SYNTHETIC:
    drift_frame, ground_truth, predictions = synthetic_inputs()
    manifest = {
        'shared_pre_frame_state': True, 'run_id': 'synthetic-smoke',
        'canonical_state_advance_action': 'L24',
    }
else:
    manifest = load_json(CFG['manifest'])
    prediction_paths = validate_manifest(manifest)
    predictions = {depth: load_predictions(path) for depth, path in prediction_paths.items()}
    prediction_tokens = validate_token_coverage(predictions)
    drift_frame = load_drift(CFG['drift_csv'], prediction_tokens)
    analysis_tokens = set(drift_frame['token'])
    ground_truth = load_ground_truth(CFG['data_root'], CFG['version'], analysis_tokens)

validate_token_coverage(predictions)
per_frame = assign_quantiles(per_frame_errors(drift_frame, ground_truth, predictions))
quantile_summary, correlation_summary = summarize_endpoint(per_frame)
box_quantile_summary, box_correlation_summary = summarize_endpoint(per_frame, 'box_degradation')

all_quantile_summaries = [quantile_summary, box_quantile_summary]
all_correlation_summaries = [correlation_summary, box_correlation_summary]
if not SYNTHETIC and Path(CFG['loss_csv']).is_file():
    losses = pd.read_csv(CFG['loss_csv'])
    required = {'token', 'loss_L8', 'loss_L12', 'loss_L16', 'loss_L24'}
    missing = required - set(losses.columns)
    if missing or losses['token'].duplicated().any():
        raise ValueError(f'invalid detector-loss table; missing={sorted(missing)}')
    per_frame = per_frame.merge(losses[list(required)], on='token', how='left', validate='one_to_one')
    if per_frame[list(required - {'token'})].isna().any().any():
        raise ValueError('detector-loss table does not cover every analysis token')
    for depth in CFG['depths'][:-1]:
        per_frame[f'loss_degradation_{depth}'] = per_frame[f'loss_{depth}'] - per_frame['loss_L24']
    loss_quantiles, loss_correlations = summarize_endpoint(per_frame, 'loss_degradation')
    all_quantile_summaries.append(loss_quantiles)
    all_correlation_summaries.append(loss_correlations)
else:
    print('Optional detector-loss table not found; skipping detector-loss sensitivity.')

quantile_summary_all = pd.concat(all_quantile_summaries, ignore_index=True)
correlation_summary_all = pd.concat(all_correlation_summaries, ignore_index=True)
correlation_summary_all

FileNotFoundError: /home/moshekimhi_google_com/StreamPETR/outputs/drift_depth_analysis/counterfactual_manifest.json

In [ ]:
per_frame.to_csv(OUTPUT_DIR / 'per_frame_degradation.csv', index=False)
quantile_summary_all.to_csv(OUTPUT_DIR / 'quantile_summary.csv', index=False)
correlation_summary_all.to_csv(OUTPUT_DIR / 'correlation_summary.csv', index=False)

resolved_cfg = {key: str(value) if isinstance(value, Path) else value for key, value in CFG.items()}
(OUTPUT_DIR / 'analysis_config.json').write_text(json.dumps(resolved_cfg, indent=2))
(OUTPUT_DIR / 'validation_summary.json').write_text(json.dumps({
    'run_id': manifest['run_id'],
    'canonical_state_advance_action': manifest['canonical_state_advance_action'],
    'shared_pre_frame_state': manifest['shared_pre_frame_state'],
    'frames': len(per_frame),
    'scenes': int(per_frame['scene_token'].nunique()),
}, indent=2))

plot_quantiles(quantile_summary, correlation_summary,
               OUTPUT_DIR / 'fig_drift_vs_depth_degradation',
               'Matched-prediction degradation versus L24')
plot_scatter(per_frame, quantile_summary,
             OUTPUT_DIR / 'fig_drift_vs_depth_degradation_scatter')
plot_quantiles(box_quantile_summary, box_correlation_summary,
               OUTPUT_DIR / 'fig_drift_vs_box_degradation',
               'Box-only degradation versus L24')
if 'loss_quantiles' in globals():
    plot_quantiles(loss_quantiles, loss_correlations,
                   OUTPUT_DIR / 'fig_drift_vs_loss_degradation',
                   'Detector-loss degradation versus L24')
if not SYNTHETIC:
    plot_quantiles(quantile_summary, correlation_summary,
                   PAPER_DIR / 'fig_drift_vs_depth_degradation',
                   'Matched-prediction degradation versus L24')
    plot_scatter(per_frame, quantile_summary,
                 PAPER_DIR / 'fig_drift_vs_depth_degradation_scatter')

In [ ]:
expected = [
    OUTPUT_DIR / 'per_frame_degradation.csv',
    OUTPUT_DIR / 'quantile_summary.csv',
    OUTPUT_DIR / 'correlation_summary.csv',
    OUTPUT_DIR / 'analysis_config.json',
    OUTPUT_DIR / 'validation_summary.json',
    OUTPUT_DIR / 'fig_drift_vs_depth_degradation.pdf',
    OUTPUT_DIR / 'fig_drift_vs_depth_degradation.png',
    OUTPUT_DIR / 'fig_drift_vs_depth_degradation_scatter.pdf',
    OUTPUT_DIR / 'fig_drift_vs_depth_degradation_scatter.png',
    OUTPUT_DIR / 'fig_drift_vs_box_degradation.pdf',
    OUTPUT_DIR / 'fig_drift_vs_box_degradation.png',
]
missing = [str(path) for path in expected if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise RuntimeError(f'missing or empty outputs: {missing}')
print(f'Validated {len(per_frame)} frames from {per_frame.scene_token.nunique()} scenes; '
      f'wrote {len(expected)} required artifacts.')